In [4]:
from data import stratified_split, WignerDataset
from pathlib import Path
import numpy as np
import h5py

In [23]:
# ------------------ INIT ZMIENNYCH -> WAŻNE!!! ------------------

# ścieżka do danych czystych i zaszumionych (dużo kłopotów)
project_dir = Path.cwd().parent.parent

noisy_path = project_dir / f"data/train_noisy_3500.h5"
print(f"Path to noisy dataset: {noisy_path}")

clean_path = project_dir / f"data/train_clean_3500.h5" 
print(f"Path to clean dataset: {clean_path}")

# ------------------ INIT ZMIENNYCH -> WAŻNE!!! ------------------

Path to noisy dataset: C:\Users\John\qsdl\data\train_noisy_350.h5
Path to clean dataset: C:\Users\John\qsdl\data\train_clean_3500.h5


In [24]:
# ------------------ INIT ZMIENNYCH -> WAŻNE!!! ------------------

# liczba wignerów w próbkach, najlepiej aby było po równo dla poprawniejszych wyników

with h5py.File(noisy_path, 'r') as file: noisy_len = file["wigner"].shape[0]
print(f"Number of noisy samples: {noisy_len}")


with h5py.File(clean_path, 'r') as file: clean_len = file["wigner"].shape[0]
print(f"Number of clean samples: {clean_len}")

# checklista czy długości są równe, zakomentuj jeśli mają być nierówne
assert noisy_len == clean_len

# ------------------ INIT ZMIENNYCH -> WAŻNE!!! ------------------

Number of noisy samples: 350


In [ ]:
train, test, val = 80, 10, 10 
rng = 67

# clean index split
trc,tec,vac = stratified_split(clean_len, rng, train, test, val)

# noisy index split
trn,ten,van = stratified_split(noisy_len, rng, train, test, val)

train_ds_noisy = WignerDataset(noisy_path, indices=trn)
test_ds_noisy = WignerDataset(noisy_path, indices=ten)
val_ds_noisy = WignerDataset(noisy_path, indices=van)

train_ds_clean = WignerDataset(clean_path, indices=trc)
test_ds_clean = WignerDataset(clean_path, indices=tec)
val_ds_clean = WignerDataset(clean_path, indices=vac)

In [ ]:
from torch.utils.data import DataLoader 

train_loader_clean = DataLoader(train_ds_clean, batch_size=64, shuffle=True)
test_loader_clean = DataLoader(test_ds_clean, batch_size=64, shuffle=False)
val_loader_clean = DataLoader(val_ds_clean, batch_size=64, shuffle=False)

train_loader_noisy = DataLoader(train_ds_noisy, batch_size=64, shuffle=True)
test_loader_noisy = DataLoader(test_ds_noisy, batch_size=64, shuffle=False)
val_loader_noisy = DataLoader(val_ds_noisy, batch_size=64, shuffle=False)


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from training import CNN_training
from classifier import StateClassifierCNN

model = StateClassifierCNN()
# ---- CLEAN ----
CNN_training(model, train_loader_clean, val_loader_clean, device, name="clean", n_epochs=60)

In [ ]:
model = StateClassifierCNN()
# ---- NOISY ----
CNN_training(model, train_loader_noisy, val_loader_noisy, device, name="noisy", n_epochs=60)

In [ ]:
from evaluation import evaluate

model_c = StateClassifierCNN().to(device)
model_n = StateClassifierCNN().to(device)

home = project_dir
model_n.load_state_dict(torch.load(f"{home}/models/noisy.pt", map_location=device, weights_only=True))
model_c.load_state_dict(torch.load(f"{home}/models/clean.pt", map_location=device, weights_only=True))


# EVAL -> ACC + CONF MATRIX 
acc_noisy_on_noisy, matrix_noisy_on_noisy = evaluate(model_n, test_loader_noisy, device)
acc_noisy_on_clean, matrix_noisy_on_clean = evaluate(model_n, test_loader_clean, device)
acc_clean_on_clean, matrix_clean_on_clean = evaluate(model_c, test_loader_clean, device)
acc_clean_on_noisy, matrix_clean_on_noisy = evaluate(model_c, test_loader_noisy, device)

In [ ]:
def plot_cm(cm, tit, filename=None):
    import matplotlib.pyplot as plt
    import numpy as np

    # names = lista 7 nazw w tej samej kolejności co LABEL_TO_ID
    names = ["fock", "coherent", "vacuum", "thermal", "cat", "gkp", "binomial"]

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.046)

    ax.set(
        xticks=np.arange(len(names)),
        yticks=np.arange(len(names)),
        xticklabels=names,
        yticklabels=names,
        ylabel="Prawdziwa klasa",
        xlabel="Predykcja",
        title=tit,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    # liczby w kratkach
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, int(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=10,
            )

    fig.tight_layout()
    if filename:
        fig.savefig(f"{filename}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [25]:
plot_cm(matrix_noisy_on_noisy, tit="Noisy model eval on noisy data", filename="n_on_n")
print(acc_noisy_on_noisy)
plot_cm(matrix_noisy_on_clean, tit="Noisy model eval on clean data", filename="n_on_c")
print(acc_noisy_on_clean)
plot_cm(matrix_clean_on_clean, tit="Clean model eval on clean data", filename="c_on_c")
print(acc_clean_on_clean)
plot_cm(matrix_clean_on_noisy, tit="Clean model eval on noisy data", filename="c_on_n")
print(acc_clean_on_noisy)

NameError: name 'plot_cm' is not defined